In [1]:
import requests
import os
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
import io


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    average_precision_score
)
import xgboost as xgb
import sys
if 'torch' in sys.modules:
    del sys.modules['torch']

import torch
import torch.autograd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
url = 'https://drive.google.com/uc?id=1r7avcqz1wm7_2NYb_gCraCiPYWZC1AxK'
df = pd.read_csv(url)
df.index = df.index + 1
df = df.drop(columns=['Id'])
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
1,6,148,72,35,0,33.6,0.627,50,1
2,1,85,66,29,0,26.6,0.351,31,0
3,8,183,64,0,0,23.3,0.672,32,1
4,1,89,66,23,94,28.1,0.167,21,0
5,0,137,40,35,168,43.1,2.288,33,1


In [3]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_cols:
    zero_count = (df[col] == 0).sum()
    print(f"{col}: {zero_count}zeros ({zero_count/len(df)*100:.1f}%)")

Glucose: 18zeros (0.7%)
BloodPressure: 125zeros (4.5%)
SkinThickness: 800zeros (28.9%)
Insulin: 1330zeros (48.0%)
BMI: 39zeros (1.4%)


In [12]:
df_clean = df.copy()
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, np.nan)
    
for col in zero_cols:
    median_non_diabetic = df_clean[df_clean["Outcome"] == 0][col].median()
    median_diabetic = df_clean[df_clean["Outcome"] == 1][col].median()
    df_clean.loc[(df_clean["Outcome"] == 0) & (df_clean[col].isna()), col] = median_non_diabetic
    df_clean.loc[(df_clean["Outcome"] == 1) & (df_clean[col].isna()), col] = median_diabetic

for col in zero_cols:
    zero_count = (df_clean[col] == 0).sum()
    print(f"  {col}: {zero_count} zeros")

  Glucose: 0 zeros
  BloodPressure: 0 zeros
  SkinThickness: 0 zeros
  Insulin: 0 zeros
  BMI: 0 zeros


In [13]:
# Feature Engineering
df_clean["Glucose_BMI_Interaction"] = df_clean["Glucose"] * df_clean["BMI"]
df_clean["Age_BMI_Interaction"] = df_clean["Age"] * df_clean["BMI"]
df_clean["Insulin_Glucose_Ratio"] = df_clean["Insulin"] / (df_clean["Glucose"] + 1)
df_clean["Metabolic_Score"] = (df_clean["Glucose"] / df_clean["Glucose"].max()) + \
                               (df_clean["BMI"] / df_clean["BMI"].max()) + \
                               (df_clean["Age"] / df_clean["Age"].max())


In [14]:
feature_cols_original = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
]
feature_cols_extended = feature_cols_original + [
    "Glucose_BMI_Interaction", "Age_BMI_Interaction",
    "Insulin_Glucose_Ratio", "Metabolic_Score"
]

X = df_clean[feature_cols_extended].values
y = df_clean["Outcome"].values


In [15]:
# metrics
RANDOM_STATE = 42
TEST_SIZE = 0.20          
VALIDATION_SIZE = 0.125   

# Neural network configs
NN_EPOCHS = 200
NN_BATCH_SIZE = 32
NN_LEARNING_RATE = 0.001
NN_PATIENCE = 20          # Early stopping patience

# Attention model configs
ATT_EPOCHS = 250
ATT_BATCH_SIZE = 32
ATT_LEARNING_RATE = 0.0005
ATT_PATIENCE = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
# Train validation test split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# second split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE, stratify=y_train_val
)


In [17]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [18]:
X_train_t = torch.FloatTensor(X_train_scaled)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_val_t = torch.FloatTensor(X_val_scaled)
y_val_t = torch.FloatTensor(y_val).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test_scaled)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)


In [19]:
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=NN_BATCH_SIZE, shuffle=True)

input_dim = X_train_scaled.shape[1]
print(f"  Input dimension: {input_dim} features")


  Input dimension: 12 features


In [20]:
traditional_models = {
    "Logistic Regression": LogisticRegression(
        C=1.0, penalty='l2', solver='lbfgs', max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_split=5,
        min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Support Vector Machine": SVC(
        C=1.0, kernel='rbf', gamma='scale', probability=True,
        random_state=RANDOM_STATE
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=7, weights='distance', metric='minkowski', p=2
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        min_samples_split=5, min_samples_leaf=2,
        subsample=0.8, random_state=RANDOM_STATE
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
        reg_lambda=1.0, random_state=RANDOM_STATE,
        use_label_encoder=False, eval_metric='logloss', verbosity=0
    ),
}



In [21]:
# Store results
results = {}
model_params = {}
feature_importance_dict = {}

for name, model in traditional_models.items():
    print(f"\n--- Training: {name} ---")

    # Train
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred_train = model.predict(X_train_scaled)
    y_pred_val = model.predict(X_val_scaled)
    y_pred_test = model.predict(X_test_scaled)

    # Probabilities for AUC
    y_prob_val = model.predict_proba(X_val_scaled)[:, 1]
    y_prob_test = model.predict_proba(X_test_scaled)[:, 1]

    # Metrics
    results[name] = {
        "train_accuracy": accuracy_score(y_train, y_pred_train),
        "val_accuracy": accuracy_score(y_val, y_pred_val),
        "test_accuracy": accuracy_score(y_test, y_pred_test),
        "test_precision": precision_score(y_test, y_pred_test),
        "test_recall": recall_score(y_test, y_pred_test),
        "test_f1": f1_score(y_test, y_pred_test),
        "val_auc_roc": roc_auc_score(y_val, y_prob_val),
        "test_auc_roc": roc_auc_score(y_test, y_prob_test),
        "test_avg_precision": average_precision_score(y_test, y_prob_test),
        "y_pred_test": y_pred_test,
        "y_prob_test": y_prob_test,
    }

    # Extract model parameters
    params = model.get_params()
    model_params[name] = params

    # Feature importance (where available)
    if hasattr(model, 'feature_importances_'):
        feature_importance_dict[name] = dict(zip(feature_cols_extended, model.feature_importances_))
    elif name == "Logistic Regression":
        feature_importance_dict[name] = dict(zip(feature_cols_extended, np.abs(model.coef_[0])))

    print(f"  Train Acc: {results[name]['train_accuracy']:.4f}")
    print(f"  Val   Acc: {results[name]['val_accuracy']:.4f}")
    print(f"  Test  Acc: {results[name]['test_accuracy']:.4f}")
    print(f"  Test  F1:  {results[name]['test_f1']:.4f}")
    print(f"  Test  AUC: {results[name]['test_auc_roc']:.4f}")


--- Training: Logistic Regression ---


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


  Train Acc: 0.7765
  Val   Acc: 0.8231
  Test  Acc: 0.7780
  Test  F1:  0.6516
  Test  AUC: 0.8622

--- Training: Random Forest ---
  Train Acc: 0.9974
  Val   Acc: 0.9819
  Test  Acc: 0.9801
  Test  F1:  0.9717
  Test  AUC: 0.9985

--- Training: Support Vector Machine ---


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  Train Acc: 0.9174
  Val   Acc: 0.9097
  Test  Acc: 0.8953
  Test  F1:  0.8490
  Test  AUC: 0.9462

--- Training: K-Nearest Neighbors ---
  Train Acc: 1.0000
  Val   Acc: 0.9856
  Test  Acc: 0.9856
  Test  F1:  0.9793
  Test  AUC: 0.9970

--- Training: Gradient Boosting ---
  Train Acc: 1.0000
  Val   Acc: 0.9892
  Test  Acc: 0.9892
  Test  F1:  0.9845
  Test  AUC: 0.9996

--- Training: XGBoost ---
  Train Acc: 0.9995
  Val   Acc: 0.9928
  Test  Acc: 0.9892
  Test  F1:  0.9844
  Test  AUC: 0.9989


In [22]:
class DiabetesNN(nn.Module):
    def __init__(self, input_dim):
        super(DiabetesNN, self).__init__()
        self.bn0 = nn.BatchNorm1d(input_dim)
        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, 32)
        self.bn3 = nn.BatchNorm1d(32)
        self.fc4 = nn.Linear(32, 16)
        self.bn4 = nn.BatchNorm1d(16)
        self.output = nn.Linear(16, 1)

    def forward(self, x):
        x = self.bn0(x)
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = torch.relu(self.bn3(self.fc3(x)))
        x = torch.relu(self.bn4(self.fc4(x)))
        x = torch.sigmoid(self.output(x))
        return x


In [23]:
def train_nn_model(model, train_loader, X_val_t, y_val_t, epochs, lr, patience, model_name):
    """Train a neural network with early stopping based on validation loss."""
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6
    )

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    print(f"\n  Training {model_name} for up to {epochs} epochs...")
    print(f"  {'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'LR':>10}")
    print(f"  {'-'*6}-+-{'-'*10}-+-{'-'*10}-+-{'-'*8}-+-{'-'*10}")

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_t)
            val_loss = criterion(val_outputs, y_val_t).item()
            val_preds = (val_outputs >= 0.5).float()
            val_acc = (val_preds == y_val_t).float().mean().item()

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if epoch % 25 == 0 or epoch == 1:
            print(f"  {epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {current_lr:>10.6f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch} (patience={patience})")
                break

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    print(f"  Best validation loss: {best_val_loss:.4f}")
    return model, history


In [24]:
nn_model = DiabetesNN(input_dim).to(DEVICE)
# Move data to device
X_train_t_dev = X_train_t.to(DEVICE)
y_train_t_dev = y_train_t.to(DEVICE)
X_val_t_dev = X_val_t.to(DEVICE)
y_val_t_dev = y_val_t.to(DEVICE)
X_test_t_dev = X_test_t.to(DEVICE)
y_test_t_dev = y_test_t.to(DEVICE)


In [25]:
train_loader_dev = DataLoader(
    TensorDataset(X_train_t_dev, y_train_t_dev),
    batch_size=NN_BATCH_SIZE, shuffle=True
)


In [26]:
nn_model, nn_history = train_nn_model(
    nn_model, train_loader_dev, X_val_t_dev, y_val_t_dev,
    epochs=NN_EPOCHS, lr=NN_LEARNING_RATE, patience=NN_PATIENCE,
    model_name="Feedforward NN (MLP)"
)



  Training Feedforward NN (MLP) for up to 200 epochs...
   Epoch | Train Loss |   Val Loss |  Val Acc |         LR
  -------+------------+------------+----------+-----------
       1 |     0.6063 |     0.4778 |   0.8520 |   0.001000
      25 |     0.2840 |     0.2073 |   0.8989 |   0.001000
      50 |     0.2759 |     0.1694 |   0.9422 |   0.001000
      75 |     0.2311 |     0.1527 |   0.9350 |   0.000500
     100 |     0.2154 |     0.1357 |   0.9458 |   0.000250
     125 |     0.2146 |     0.1293 |   0.9458 |   0.000125
     150 |     0.2044 |     0.1247 |   0.9458 |   0.000125
  Early stopping at epoch 165 (patience=20)
  Best validation loss: 0.1197


In [27]:
nn_model.eval()
with torch.no_grad():
    nn_train_prob = nn_model(X_train_t_dev).cpu().numpy().flatten()
    nn_val_prob = nn_model(X_val_t_dev).cpu().numpy().flatten()
    nn_test_prob = nn_model(X_test_t_dev).cpu().numpy().flatten()

In [28]:
nn_train_pred = (nn_train_prob >= 0.5).astype(int)
nn_val_pred = (nn_val_prob >= 0.5).astype(int)
nn_test_pred = (nn_test_prob >= 0.5).astype(int)


In [29]:
results["Neural Network (MLP)"] = {
    "train_accuracy": accuracy_score(y_train, nn_train_pred),
    "val_accuracy": accuracy_score(y_val, nn_val_pred),
    "test_accuracy": accuracy_score(y_test, nn_test_pred),
    "test_precision": precision_score(y_test, nn_test_pred),
    "test_recall": recall_score(y_test, nn_test_pred),
    "test_f1": f1_score(y_test, nn_test_pred),
    "val_auc_roc": roc_auc_score(y_val, nn_val_prob),
    "test_auc_roc": roc_auc_score(y_test, nn_test_prob),
    "test_avg_precision": average_precision_score(y_test, nn_test_prob),
    "y_pred_test": nn_test_pred,
    "y_prob_test": nn_test_prob,
}


In [30]:
nn_param_count = sum(p.numel() for p in nn_model.parameters())
nn_trainable_count = sum(p.numel() for p in nn_model.parameters() if p.requires_grad)


In [31]:
nn_params_summary = {
    "Architecture": "Input->BN->128->BN->Drop(0.3)->64->BN->Drop(0.3)->32->BN->16->BN->1(Sigmoid)",
    "Input Dimension": input_dim,
    "Total Parameters": nn_param_count,
    "Trainable Parameters": nn_trainable_count,
    "Optimizer": "Adam (weight_decay=1e-4)",
    "Learning Rate": NN_LEARNING_RATE,
    "Scheduler": "ReduceLROnPlateau (factor=0.5, patience=10)",
    "Loss Function": "BCELoss",
    "Batch Size": NN_BATCH_SIZE,
    "Max Epochs": NN_EPOCHS,
    "Early Stopping Patience": NN_PATIENCE,
    "Dropout Rate": 0.3,
    "Activation": "ReLU (hidden), Sigmoid (output)",
    "Batch Normalization": "Yes (all hidden layers)",
}


In [32]:
model_params["Neural Network (MLP)"] = nn_params_summary


In [33]:
print(f"\n  MLP Test Results:")
print(f"    Accuracy:  {results['Neural Network (MLP)']['test_accuracy']:.4f}")
print(f"    F1 Score:  {results['Neural Network (MLP)']['test_f1']:.4f}")
print(f"    AUC-ROC:   {results['Neural Network (MLP)']['test_auc_roc']:.4f}")
print(f"    Parameters: {nn_param_count} total, {nn_trainable_count} trainable")



  MLP Test Results:
    Accuracy:  0.9368
    F1 Score:  0.9109
    AUC-ROC:   0.9837
    Parameters: 13049 total, 13049 trainable


In [35]:
class SelfAttentionLayer(nn.Module):
    def __init__(self, feature_dim, num_heads=4):
        super(SelfAttentionLayer, self).__init__()
        self.num_heads = num_heads
        self.feature_dim = feature_dim
        self.head_dim = feature_dim // num_heads
        assert feature_dim % num_heads == 0, "feature_dim must be divisible by num_heads"

        self.W_q = nn.Linear(feature_dim, feature_dim)
        self.W_k = nn.Linear(feature_dim, feature_dim)
        self.W_v = nn.Linear(feature_dim, feature_dim)
        self.W_o = nn.Linear(feature_dim, feature_dim)
        self.scale = torch.sqrt(torch.FloatTensor([self.head_dim]))

    def forward(self, x):
        batch_size = x.shape[0]
        # x: (batch, feature_dim)
        # Reshape for attention: treat as (batch, 1, feature_dim) — single "token"
        x = x.unsqueeze(1)  # (batch, 1, feature_dim)

        Q = self.W_q(x)  # (batch, 1, feature_dim)
        K = self.W_k(x)
        V = self.W_v(x)

        # Multi-head split
        Q = Q.view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale.to(x.device)
        attention_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(attention_weights, V)

        # Concatenate heads
        context = context.transpose(1, 2).contiguous().view(batch_size, 1, self.feature_dim)
        output = self.W_o(context).squeeze(1)  # (batch, feature_dim)
        return output, attention_weights




In [36]:
class FeatureAttentionLayer(nn.Module):
    def __init__(self, input_dim, attention_dim=64):
        super(FeatureAttentionLayer, self).__init__()
        self.feature_embeddings = nn.Linear(input_dim, attention_dim)
        self.query = nn.Parameter(torch.randn(attention_dim))
        self.key = nn.Linear(attention_dim, attention_dim)
        self.value = nn.Linear(attention_dim, input_dim)
        self.scale = torch.sqrt(torch.FloatTensor([attention_dim]))

    def forward(self, x):
        # x: (batch, input_dim)
        embeddings = torch.relu(self.feature_embeddings(x))  # (batch, attention_dim)
        keys = self.key(embeddings)  # (batch, attention_dim)

        # Compute attention: query (attention_dim,) dot keys (batch, attention_dim)
        scores = torch.matmul(keys, self.query) / self.scale.to(x.device)  # (batch,)
        # But we want feature-level attention, so let's do it differently

        # Feature-level attention
        # Reshape x to (batch, input_dim, 1) and compute per-feature attention
        feat_scores = torch.matmul(embeddings, self.query)  # (batch,)
        # Use a different approach: compute attention per feature group
        # We project each feature through embedding and compute importance
        attention_logits = torch.matmul(
            embeddings.unsqueeze(1),   # (batch, 1, attention_dim)
            self.query.unsqueeze(0).unsqueeze(2)  # (1, attention_dim, 1)
        ).squeeze(-1)  # (batch, 1)

        # Actually, let's use a cleaner feature-attention mechanism
        return x, None  # placeholder, will be overridden below


In [37]:
class AttentionDiabetesNN(nn.Module):
    def __init__(self, input_dim, d_model=64, num_heads=4, dropout=0.3):
        super(AttentionDiabetesNN, self).__init__()

        # Feature embedding
        self.feature_embed = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.BatchNorm1d(d_model),
            nn.ReLU(),
        )

        # Multi-head self-attention
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.scale = torch.sqrt(torch.FloatTensor([self.head_dim]))

        # Feature gating (channel attention)
        self.gate = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.ReLU(),
            nn.Linear(d_model // 4, d_model),
            nn.Sigmoid(),
        )

        # Feed-forward network after attention
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.BatchNorm1d(d_model * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.BatchNorm1d(d_model),
        )

        # Layer normalization
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(16, 1),
        )

        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def multi_head_attention(self, x):
        """Multi-head self-attention for feature interaction modeling."""
        batch_size = x.shape[0]
        # Treat the feature vector as a single token
        x = x.unsqueeze(1)  # (batch, 1, d_model)

        Q = self.W_q(x).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale.to(x.device)
        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)

        context = context.transpose(1, 2).contiguous().view(batch_size, 1, self.d_model)
        output = self.W_o(context).squeeze(1)  # (batch, d_model)
        return output, attn_weights

    def forward(self, x):
        # 1. Feature embedding
        embedded = self.feature_embed(x)  # (batch, d_model)

        # 2. Multi-head self-attention with residual connection
        attn_output, attn_weights = self.multi_head_attention(embedded)
        attn_output = self.dropout(attn_output)
        norm1 = self.ln1(embedded + attn_output)  # Residual + LayerNorm

        # 3. Feature gating (channel attention)
        gate_weights = self.gate(norm1)  # (batch, d_model), values in [0,1]
        gated = norm1 * gate_weights  # Element-wise gating

        # 4. FFN with residual connection
        ffn_output = self.ffn(gated)
        ffn_output = self.dropout(ffn_output)
        norm2 = self.ln2(gated + ffn_output)  # Residual + LayerNorm

        # 5. Classification
        logits = self.classifier(norm2)
        output = torch.sigmoid(logits)
        return output, attn_weights, gate_weights



In [38]:
att_model = AttentionDiabetesNN(
    input_dim=input_dim, d_model=64, num_heads=4, dropout=0.3
).to(DEVICE)

att_param_count = sum(p.numel() for p in att_model.parameters())
att_trainable_count = sum(p.numel() for p in att_model.parameters() if p.requires_grad)
print(f"  Total parameters: {att_param_count}, Trainable: {att_trainable_count}")


  Total parameters: 39665, Trainable: 39665


In [39]:
criterion = nn.BCELoss()
optimizer_att = optim.AdamW(att_model.parameters(), lr=ATT_LEARNING_RATE, weight_decay=1e-3)
scheduler_att = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_att, T_0=20, T_mult=2
)


In [40]:
best_val_loss = float('inf')
best_att_state = None
patience_counter = 0
att_history = {"train_loss": [], "val_loss": [], "val_acc": []}


In [41]:
for epoch in range(1, ATT_EPOCHS + 1):
    att_model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader_dev:
        optimizer_att.zero_grad()
        outputs, _, _ = att_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(att_model.parameters(), max_norm=1.0)
        optimizer_att.step()
        train_loss += loss.item()

    train_loss /= len(train_loader_dev)
    scheduler_att.step()

    # Validation
    att_model.eval()
    with torch.no_grad():
        val_outputs, _, _ = att_model(X_val_t_dev)
        val_loss = criterion(val_outputs, y_val_t_dev).item()
        val_preds = (val_outputs >= 0.5).float()
        val_acc = (val_preds == y_val_t_dev).float().mean().item()

    current_lr = optimizer_att.param_groups[0]['lr']
    att_history["train_loss"].append(train_loss)
    att_history["val_loss"].append(val_loss)
    att_history["val_acc"].append(val_acc)

    if epoch % 25 == 0 or epoch == 1:
        print(f"  {epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {current_lr:>10.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_att_state = att_model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= ATT_PATIENCE:
            print(f"  Early stopping at epoch {epoch} (patience={ATT_PATIENCE})")
            break



       1 |     0.5723 |     0.4353 |   0.8231 |   0.000497
      25 |     0.2951 |     0.2123 |   0.9206 |   0.000481
      50 |     0.2160 |     0.1621 |   0.9350 |   0.000073
      75 |     0.1990 |     0.1410 |   0.9422 |   0.000458
     100 |     0.1675 |     0.1060 |   0.9495 |   0.000250
     125 |     0.1486 |     0.0861 |   0.9639 |   0.000042
  Early stopping at epoch 146 (patience=25)


In [42]:
if best_att_state is not None:
    att_model.load_state_dict(best_att_state)

print(f"  Best validation loss: {best_val_loss:.4f}")

  Best validation loss: 0.0750


In [43]:
att_model.eval()
with torch.no_grad():
    att_train_prob, _, att_train_gate = att_model(X_train_t_dev)
    att_val_prob, _, att_val_gate = att_model(X_val_t_dev)
    att_test_prob, _, att_test_gate = att_model(X_test_t_dev)

    att_train_prob = att_train_prob.cpu().numpy().flatten()
    att_val_prob = att_val_prob.cpu().numpy().flatten()
    att_test_prob = att_test_prob.cpu().numpy().flatten()
    att_test_gate = att_test_gate.cpu().numpy()


In [44]:
att_train_pred = (att_train_prob >= 0.5).astype(int)
att_val_pred = (att_val_prob >= 0.5).astype(int)
att_test_pred = (att_test_prob >= 0.5).astype(int)


In [45]:
results["Attention-Based NN"] = {
    "train_accuracy": accuracy_score(y_train, att_train_pred),
    "val_accuracy": accuracy_score(y_val, att_val_pred),
    "test_accuracy": accuracy_score(y_test, att_test_pred),
    "test_precision": precision_score(y_test, att_test_pred),
    "test_recall": recall_score(y_test, att_test_pred),
    "test_f1": f1_score(y_test, att_test_pred),
    "val_auc_roc": roc_auc_score(y_val, att_val_prob),
    "test_auc_roc": roc_auc_score(y_test, att_test_prob),
    "test_avg_precision": average_precision_score(y_test, att_test_prob),
    "y_pred_test": att_test_pred,
    "y_prob_test": att_test_prob,
}


In [46]:
att_params_summary = {
    "Architecture": "FeatureEmbed(d_model=64) -> MultiHead-Attention(4 heads) "
                    "-> Residual+LayerNorm -> FeatureGate(Sigmoid) -> FFN(128) "
                    "-> Residual+LayerNorm -> Classifier(64->32->16->1)",
    "Input Dimension": input_dim,
    "d_model (Embedding Dim)": 64,
    "Number of Attention Heads": 4,
    "Head Dimension": 16,
    "FFN Hidden Dim": 128,
    "Total Parameters": att_param_count,
    "Trainable Parameters": att_trainable_count,
    "Optimizer": "AdamW (weight_decay=1e-3)",
    "Learning Rate": ATT_LEARNING_RATE,
    "Scheduler": "CosineAnnealingWarmRestarts (T_0=20, T_mult=2)",
    "Loss Function": "BCELoss",
    "Gradient Clipping": "max_norm=1.0",
    "Batch Size": ATT_BATCH_SIZE,
    "Max Epochs": ATT_EPOCHS,
    "Early Stopping Patience": ATT_PATIENCE,
    "Dropout Rate": 0.3,
    "Attention Type": "Multi-Head Self-Attention + Feature Gating (Channel Attention)",
    "Residual Connections": "Yes (post-attention + post-FFN)",
    "Layer Normalization": "Yes (post-attention + post-FFN)",
    "Activation": "ReLU (hidden), Sigmoid (output + gate)",
}


In [47]:
confusion_matrices = {}
for name, res in results.items():
    cm = confusion_matrix(y_test, res["y_pred_test"])
    confusion_matrices[name] = cm
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    results[name]["test_specificity"] = specificity
    results[name]["test_npv"] = npv
    print(f"\n  {name}:")
    print(f"    TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(f"    Specificity: {specificity:.4f}, NPV: {npv:.4f}")



  Logistic Regression:
    TN=316, FP=47, FN=76, TP=115
    Specificity: 0.8705, NPV: 0.8061

  Random Forest:
    TN=354, FP=9, FN=2, TP=189
    Specificity: 0.9752, NPV: 0.9944

  Support Vector Machine:
    TN=333, FP=30, FN=28, TP=163
    Specificity: 0.9174, NPV: 0.9224

  K-Nearest Neighbors:
    TN=357, FP=6, FN=2, TP=189
    Specificity: 0.9835, NPV: 0.9944

  Gradient Boosting:
    TN=357, FP=6, FN=0, TP=191
    Specificity: 0.9835, NPV: 1.0000

  XGBoost:
    TN=359, FP=4, FN=2, TP=189
    Specificity: 0.9890, NPV: 0.9945

  Neural Network (MLP):
    TN=340, FP=23, FN=12, TP=179
    Specificity: 0.9366, NPV: 0.9659

  Attention-Based NN:
    TN=356, FP=7, FN=7, TP=184
    Specificity: 0.9807, NPV: 0.9807


In [48]:
cv_results = {}
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in traditional_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='accuracy')
    cv_results[name] = {
        "cv_mean_accuracy": scores.mean(),
        "cv_std_accuracy": scores.std(),
        "cv_scores": scores.tolist(),
    }


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.1

In [49]:
perf_data = []
for name, res in results.items():
    perf_data.append({
        "Model": name,
        "Train Accuracy": round(res["train_accuracy"], 4),
        "Validation Accuracy": round(res["val_accuracy"], 4),
        "Test Accuracy": round(res["test_accuracy"], 4),
        "Test Precision": round(res["test_precision"], 4),
        "Test Recall (Sensitivity)": round(res["test_recall"], 4),
        "Test Specificity": round(res["test_specificity"], 4),
        "Test F1 Score": round(res["test_f1"], 4),
        "Validation AUC-ROC": round(res["val_auc_roc"], 4),
        "Test AUC-ROC": round(res["test_auc_roc"], 4),
        "Test Avg Precision (AUC-PR)": round(res["test_avg_precision"], 4),
        "Test NPV": round(res["test_npv"], 4),
    })

perf_df = pd.DataFrame(perf_data)
perf_df = perf_df.sort_values("Test AUC-ROC", ascending=False).reset_index(drop=True)
perf_df.index = perf_df.index + 1
perf_df.index.name = "Rank"


In [50]:
params_rows = []
for model_name, params in model_params.items():
    if isinstance(params, dict) and any(isinstance(v, (int, float, str, bool)) for v in params.values()):
        # Neural network params (clean dict)
        for param_name, param_value in params.items():
            params_rows.append({
                "Model": model_name,
                "Parameter": param_name,
                "Value": str(param_value),
            })
    else:
        # Sklearn/xgboost params (get_params dict)
        for param_name, param_value in sorted(params.items()):
            params_rows.append({
                "Model": model_name,
                "Parameter": param_name,
                "Value": str(param_value),
            })

params_df = pd.DataFrame(params_rows)


In [51]:
cm_rows = []
for model_name, cm in confusion_matrices.items():
    tn, fp, fn, tp = cm.ravel()
    cm_rows.append({
        "Model": model_name,
        "True Negative (TN)": int(tn),
        "False Positive (FP)": int(fp),
        "False Negative (FN)": int(fn),
        "True Positive (TP)": int(tp),
        "Total Samples": int(tn + fp + fn + tp),
    })

cm_df = pd.DataFrame(cm_rows)



In [56]:
import openpyxl
fi_rows = []
for model_name, fi_dict in feature_importance_dict.items():
    for feat, importance in fi_dict.items():
        fi_rows.append({
            "Model": model_name,
            "Feature": feat,
            "Importance": round(importance, 6),
        })

fi_df = pd.DataFrame(fi_rows)

# ---- Excel File 5: Cross-Validation Results ----
cv_rows = []
for model_name, cv_res in cv_results.items():
    cv_rows.append({
        "Model": model_name,
        "CV Mean Accuracy": round(cv_res["cv_mean_accuracy"], 4),
        "CV Std Accuracy": round(cv_res["cv_std_accuracy"], 4),
        "Fold 1": round(cv_res["cv_scores"][0], 4),
        "Fold 2": round(cv_res["cv_scores"][1], 4),
        "Fold 3": round(cv_res["cv_scores"][2], 4),
        "Fold 4": round(cv_res["cv_scores"][3], 4),
        "Fold 5": round(cv_res["cv_scores"][4], 4),
    })

cv_df = pd.DataFrame(cv_rows)

# ---- Excel File 6: Training History (Neural Networks) ----
max_len = max(len(nn_history["train_loss"]), len(att_history["train_loss"]))
history_rows = []
for i in range(max_len):
    row = {"Epoch": i + 1}
    if i < len(nn_history["train_loss"]):
        row["MLP Train Loss"] = round(nn_history["train_loss"][i], 6)
        row["MLP Val Loss"] = round(nn_history["val_loss"][i], 6)
        row["MLP Val Accuracy"] = round(nn_history["val_acc"][i], 6)
    if i < len(att_history["train_loss"]):
        row["Attention Train Loss"] = round(att_history["train_loss"][i], 6)
        row["Attention Val Loss"] = round(att_history["val_loss"][i], 6)
        row["Attention Val Accuracy"] = round(att_history["val_acc"][i], 6)
    history_rows.append(row)

history_df = pd.DataFrame(history_rows)

# ---- Excel File 7: Data Summary ----
data_summary = pd.DataFrame({
    "Metric": [
        "Total Samples", "Training Samples", "Validation Samples", "Test Samples",
        "Features (Original)", "Features (Engineered)", "Diabetic (Outcome=1)",
        "Non-Diabetic (Outcome=0)", "Class Imbalance Ratio",
        "Imputation Method", "Scaling Method", "Random State"
    ],
    "Value": [
        len(df), len(y_train), len(y_val), len(y_test),
        8, 12, int(y.sum()), int(len(y) - y.sum()),
        f"1:{(len(y) - y.sum()) / y.sum():.2f}",
        "Group-wise Median (by Outcome)", "StandardScaler", str(RANDOM_STATE)
    ]
})

# ---- Excel File 8: Attention Gate Weights ----
avg_gate_weights = att_test_gate.mean(axis=0)
gate_rows = []
for i, feat in enumerate(feature_cols_extended):
    gate_rows.append({
        "Feature": feat,
        "Avg Gate Weight (Test)": round(float(avg_gate_weights[i]), 4),
    })
gate_df = pd.DataFrame(gate_rows).sort_values("Avg Gate Weight (Test)", ascending=False)

# ---- Excel File 9: Classification Reports ----
report_rows = []
for name, res in results.items():
    report = classification_report(y_test, res["y_pred_test"], output_dict=True)
    for class_label, metrics in report.items():
        if isinstance(metrics, dict):
            report_rows.append({
                "Model": name,
                "Class": class_label,
                "Precision": round(metrics.get("precision", 0), 4),
                "Recall": round(metrics.get("recall", 0), 4),
                "F1-Score": round(metrics.get("f1-score", 0), 4),
                "Support": metrics.get("support", 0),
            })

report_df = pd.DataFrame(report_rows)

# ===== WRITE TO EXCEL FILES =====
# File 1: Comprehensive Performance Report
excel_path_1 = os.path.join("diabetes_model_performance.xlsx")
with pd.ExcelWriter(excel_path_1, engine='openpyxl') as writer:
    perf_df.to_excel(writer, sheet_name="Performance Comparison", index=True)
    cm_df.to_excel(writer, sheet_name="Confusion Matrices", index=False)
    report_df.to_excel(writer, sheet_name="Classification Reports", index=False)
    cv_df.to_excel(writer, sheet_name="Cross-Validation", index=False)
    data_summary.to_excel(writer, sheet_name="Data Summary", index=False)

print(f"  Saved: {excel_path_1}")
print(f"    Sheets: Performance Comparison, Confusion Matrices,")
print(f"            Classification Reports, Cross-Validation, Data Summary")

# File 2: Model Parameters
excel_path_2 = os.path.join("diabetes_model_parameters.xlsx")
with pd.ExcelWriter(excel_path_2, engine='openpyxl') as writer:
    params_df.to_excel(writer, sheet_name="All Model Parameters", index=False)
    fi_df.to_excel(writer, sheet_name="Feature Importance", index=False)
    gate_df.to_excel(writer, sheet_name="Attention Gate Weights", index=False)
    history_df.to_excel(writer, sheet_name="Training History", index=False)



  Saved: diabetes_model_performance.xlsx
    Sheets: Performance Comparison, Confusion Matrices,
            Classification Reports, Cross-Validation, Data Summary
